# J05 — Écoulement non saturé permanent et introduction à HYDRUS-1D

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 5 : fonctions hydrauliques, profils permanents, lecture d'un projet HYDRUS-1D, infiltromètre à disque*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Les projets HYDRUS-1D de référence sont dans `../../hydrus/` et se lisent avec le module `hydrus_io.py` (dossier `notebooks/`).
Unités du cours : **cm** et **jours** ; flux **positifs vers le haut** (convention HYDRUS-1D) ; profondeurs négatives.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, integrate
from hydrus_io import read_tlevel, read_nod_inf, read_obs_node, read_balance, read_run_inf

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})
HYD = "../../hydrus"

# Sols de Carsel & Parrish (1988) : thr, ths, alpha (1/cm), n, Ks (cm/j), l
SOLS = {
    "sable":         [0.045, 0.430, 0.145, 2.68, 712.8, 0.5],
    "loam sableux":  [0.065, 0.410, 0.075, 1.89, 106.1, 0.5],
    "loam":          [0.078, 0.430, 0.036, 1.56, 24.96, 0.5],
    "loam limoneux": [0.067, 0.450, 0.020, 1.41, 10.80, 0.5],
    "argile":        [0.068, 0.380, 0.008, 1.09, 4.80, 0.5],
}

## Exercice 1 — Fonctions hydrauliques de Mualem–van Genuchten  *(≈ 15 min)*

1. Écrire `vg_theta(h, p)`, `vg_K(h, p)` et `vg_C(h, p)` (vectorisées, `p = [thr, ths, alpha, n, Ks, l]`) :
   $S_e = [1+(\alpha|h|)^n]^{-m}$, $m = 1-1/n$ ; $\theta = \theta_r + (\theta_s-\theta_r)S_e$ ;
   $K = K_s S_e^{l}[1-(1-S_e^{1/m})^m]^2$ ; $C = (\theta_s-\theta_r)\,\alpha n m (\alpha|h|)^{n-1}[1+(\alpha|h|)^n]^{-m-1}$ (et 0 pour $h \ge 0$).
2. Tracer $K(h)$ et $C(h)$ en fonction de $|h|$ (échelles log-log) pour le sable, le loam et l'argile, puis $D(\theta) = K/C$ (échelle log en $y$).
3. Calculer la longueur capillaire macroscopique $\lambda_c = K_s^{-1}\int_{-\infty}^{0} K(h)\,dh$ pour les cinq sols
   (`integrate.quad` avec des points de rupture, ou trapèzes sur une grille logarithmique) et $\alpha_G = 1/\lambda_c$.
   Vérifier le loam : $K(-50) \approx 0{,}258$ cm/j, $K(-500) \approx 1{,}7\times10^{-4}$ cm/j, $\lambda_c \approx 6{,}9$ cm.

In [ ]:
def vg_theta(h, p):
    thr, ths, a, n, Ks, l = p
    m = 1 - 1 / n
    h = np.asarray(h, float)
    Se = np.where(h < 0, (1 + (a * np.abs(h))**n)**(-m), 1.0)
    return thr + (ths - thr) * Se

def vg_K(h, p):
    thr, ths, a, n, Ks, l = p
    m = 1 - 1 / n
    h = np.asarray(h, float)
    Se = np.where(h < 0, (1 + (a * np.abs(h))**n)**(-m), 1.0)
    return # À COMPLÉTER

def vg_C(h, p):
    thr, ths, a, n, Ks, l = p
    m = 1 - 1 / n
    h = np.asarray(h, float)
    return # À COMPLÉTER (0 pour h >= 0)

print("K(-50) =", vg_K(-50, SOLS["loam"]), " K(-500) =", vg_K(-500, SOLS["loam"]))

# 2. tracés log-log de K(h), C(h) et D(theta)
hh = -np.logspace(-1, 4, 300)
# À COMPLÉTER

# 3. longueur capillaire
def lambda_c(p):
    # À COMPLÉTER : quad(lambda h: vg_K(h, p) / p[4], -1e4, 0, limit=400, points=[-1000, -100, -10, -1])
    pass

## Exercice 2 — Profils permanents au-dessus d'une nappe et évaporation maximale  *(≈ 20 min)*

En régime permanent le flux $q$ est constant et $\dfrac{dh}{dz} = -1 - \dfrac{q}{K(h)}$ ($z$ vers le haut, $q > 0$ vers le haut).

1. Écrire `profil_permanent(q, p, L=100)` qui intègre cette EDO avec `integrate.solve_ivp` depuis la nappe ($h = 0$ à $z = -L$)
   jusqu'à la surface ($z = 0$), pour le loam et $q = -1$ cm/j (infiltration) puis $q = +0{,}03$ cm/j (évaporation).
   Prévoir un événement terminal si $h < -10^4$ cm (le flux ne peut pas être maintenu).
2. Lire `NOD_INF.OUT` des projets `J05_permanent_nappe_infiltration` et `J05_permanent_nappe_evaporation` (`read_nod_inf`),
   extraire le profil à $t = 200$ j et le superposer à votre intégration ; calculer l'écart RMS sur $h$ et $h$ en surface
   (attendu : $\approx -28{,}6$ cm par intégration, $-28{,}9$ cm dans HYDRUS pour l'infiltration).
3. Évaporation maximale : la profondeur de nappe maximale compatible avec un flux $q$ est
   $L_{max}(q) = \int_{-\infty}^{0} \dfrac{dh}{1 + q/K(h)}$ (Gardner 1958). Écrire `L_max(q, p)` (quadrature) puis
   `e_max(p, L)` (`optimize.brentq` sur $L_{max}(q) - L$). Calculer $e_{max}$ du loam pour $L = 100$ cm (attendu $\approx 0{,}054$ cm/j)
   et tracer $e_{max}(L)$ pour $L$ de 30 à 300 cm pour le sable, le loam et le loam limoneux (échelles log).

In [ ]:
def profil_permanent(q, p, L=100.0):
    """Intègre dh/dz = -1 - q/K(h) de z = -L (h = 0) à z = 0. Retourne z, h."""
    f = lambda z, h: [-1.0 - q / vg_K(h[0], p)]
    ev = lambda z, h: h[0] + 1e4; ev.terminal = True
    # À COMPLÉTER : sol = integrate.solve_ivp(f, [-L, 0], [0.0], max_step=0.5, rtol=1e-8, atol=1e-10, events=ev)
    pass

# 1. profils loam pour q = -1 et q = +0.03
# 2. comparaison avec HYDRUS : nod = read_nod_inf(f"{HYD}/J05_permanent_nappe_infiltration") ; nod[200.0][["Depth", "Head"]]

# 3. évaporation maximale
def L_max(q, p):
    with np.errstate(divide="ignore"):
        # À COMPLÉTER : integrate.quad(lambda h: 1/(1 + q/vg_K(h, p)), -1e7, 0, limit=500, points=[-1e5, -1e4, -1e3, -100, -10, -1])
        pass

def e_max(p, L):
    # À COMPLÉTER : optimize.brentq(lambda q: L_max(q, p) - L, 1e-12, p[4])
    pass

## Exercice 3 — Lecture d'un projet HYDRUS-1D : convergence vers le régime permanent  *(≈ 15 min)*

Les deux projets `J05_permanent_nappe_*` partent d'un profil hydrostatique ($h = -100 - z$, $H = -100$ cm) et imposent un flux constant
en surface (−1 cm/j ou +0,03 cm/j) avec une nappe fixe ($h = 0$) au bas d'une colonne de loam de 100 cm, pendant 200 j.

1. Lire `T_LEVEL.OUT` (`read_tlevel`) et tracer `vTop`, `vBot` (flux en surface et au bas, > 0 vers le haut) et `hTop`
   en fonction du temps pour les deux projets. Déterminer $t_{99}$, premier instant où $|$`vBot` − `vTop`$| < 1$ % de $|q|$.
2. Lire `RUN_INF.OUT` (`read_run_inf`) : tracer $\Delta t$ en fonction du temps (échelle log), l'histogramme du nombre
   d'itérations par pas ; compter les pas et les itérations totales.
3. Lire `BALANCE.OUT` (`read_balance`) : erreur relative maximale `WatBalR` (%) et évolution du stock `W-volume` ;
   vérifier que la variation de stock entre 0 et 200 j est cohérente avec `sum(vTop)` − `sum(vBot)` de `T_LEVEL.OUT`
   (attention aux signes : $\Delta W = \int (q_{bas} - q_{haut})\,dt$ avec $q > 0$ vers le haut).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, tag in enumerate(["infiltration", "evaporation"]):
    tl = read_tlevel(f"{HYD}/J05_permanent_nappe_{tag}")
    # À COMPLÉTER : tracés de vTop, vBot ; t99
    q = tl["vTop"].iloc[-1]
    ecart = (tl["vBot"] - tl["vTop"]).abs()
    t99 = # À COMPLÉTER
    print(tag, "t99 =", t99)

# 2. RUN_INF.OUT : ri = read_run_inf(...) ; colonnes Time, dt, Iter, ItCum
# 3. BALANCE.OUT : bal = read_balance(...) ; colonnes W-volume, WatBalT, WatBalR

## Exercice 4 — Infiltromètre à disque : Wooding et méthode à deux tensions  *(≈ 10 min)*

`data/J05_infiltrometre.csv` : pour quatre sites (textures indiquées), débits permanents $Q$ (cm³/min) mesurés avec un disque de rayon
$r = 10$ cm aux tensions $h_0 = -3$ cm et $-10$ cm.

Solution de Wooding (1968) avec $K = K_s e^{\alpha_G h}$ : $Q = \pi r^2 K(h_0)\left(1 + \dfrac{4}{\pi r \alpha_G}\right)$.

1. Pour chaque site, estimer $\alpha_G = \ln(Q_1/Q_2)/(h_1 - h_2)$, $\lambda_c = 1/\alpha_G$, puis $K_s$ (convertir $Q$ en cm³/j).
2. Calculer la part du débit due à la capillarité latérale, $4/(\pi r\alpha_G) / (1 + 4/(\pi r\alpha_G))$, à $h_0 = -3$ cm.
3. Comparer $K_s$ et $\lambda_c$ aux valeurs de Carsel & Parrish de la texture (exercice 1) ; commenter les écarts.

In [ ]:
inf = pd.read_csv("data/J05_infiltrometre.csv")
display(inf)
h1, h2 = -3.0, -10.0
Q1 = inf["Q_h3_cm3min"] * 1440      # cm3/j
Q2 = inf["Q_h10_cm3min"] * 1440
r = inf["r_cm"]
inf["alpha_G"] = # À COMPLÉTER
inf["lambda_c"] = # À COMPLÉTER
inf["Ks"] = # À COMPLÉTER
inf["part_capillaire_%"] = # À COMPLÉTER

## Exercice 5 — Bonus — solution analytique de Gardner pour l'évaporation maximale  *(≈ facultatif)*

Pour $K = K_s e^{\alpha_G h}$, montrer que $L_{max}(q) = \dfrac{1}{\alpha_G}\ln\!\left(1 + \dfrac{K_s}{q}\right)$ et donc
$e_{max}(L) = \dfrac{K_s}{e^{\alpha_G L} - 1}$. Calculer $e_{max}$ du loam ($K_s = 24{,}96$ cm/j, $\alpha_G = 1/\lambda_c$) pour
$L = 30, 50, 100$ cm et comparer aux valeurs MvG de l'exercice 2. D'où vient l'écart ?

In [ ]:
p = SOLS["loam"]
aG = 1 / lambda_c(p)
for L in [30, 50, 100]:
    e_gardner = # À COMPLÉTER
    print(L, e_gardner, e_max(p, L))

## Pour aller plus loin

* Reprendre l'exercice 2 avec la nappe à 50 cm et à 150 cm : comment évolue $h$ en surface pour $q = -1$ cm/j ?
* Dans HYDRUS-1D, remplacer le flux imposé de $+0{,}03$ cm/j par $+0{,}06$ cm/j ($> e_{max}$) et observer `RUN_INF.OUT`.
* Réaliser l'essai à trois tensions ($-3$, $-6$, $-10$ cm) et vérifier la linéarité de $\ln Q$ en fonction de $h_0$.